# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:

# Load the data 
df = pd.read_csv("data/AviationData.csv", encoding="latin-1", low_memory=False)

# Shape and data types
print(df.shape)
df.info()

# Missing values per column
display(df.isna().sum())

# Summary statistics for numeric columns
display(df.describe())

(88889, 31)
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-nul

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# --- Inspect the relevant columns ---
print(df["Aircraft.Category"].value_counts(dropna=False).head(), "\n")
print(df["Amateur.Built"].value_counts(dropna=False), "\n")

# --- Standardize Make names first (needed for the imputation below) ---
# The same manufacturer appears with different capitalization and spellings
# (e.g. "CESSNA", "Cessna", "Cessna Aircraft Co")
df["Make"] = df["Make"].str.strip().str.title()
make_map = {
    "Air Tractor Inc": "Air Tractor", "Air Tractor Inc.": "Air Tractor",
    "Cirrus Design Corp": "Cirrus", "Cirrus Design Corp.": "Cirrus",
    "Aviat Aircraft Inc": "Aviat", "Diamond Aircraft Ind Inc": "Diamond",
    "Diamond Aircraft Industries": "Diamond", "Dehavilland": "De Havilland",
    "Ayres Corporation": "Ayres", "American Champion Aircraft": "Champion",
    "American Champion": "Champion", "Raytheon Aircraft Company": "Beech",
    "Hawker Beechcraft": "Beech", "Hawker Beechcraft Corp": "Beech", "Beechcraft": "Beech",
    "Rockwell International": "Rockwell", "Grumman Acft Eng Cor-Schweizer": "Grumman",
    "Grumman American Avn. Corp.": "Grumman", "Grumman American": "Grumman",
    "Grumman-Schweizer": "Grumman", "Mooney Aircraft Corp.": "Mooney",
    "Piper Aircraft Inc": "Piper", "The New Piper Aircraft": "Piper",
    "New Piper Aircraft Inc": "Piper", "Piper Aircraft Corporation": "Piper",
    "Cessna Aircraft Co": "Cessna", "Textron Aviation Inc": "Cessna",
    "Flight Design Gmbh": "Flight Design", "Embraer S A": "Embraer",
    "Airbus Industrie": "Airbus", "Bombardier Inc": "Bombardier",
    "Bombardier Inc.": "Bombardier", "Canadair": "Bombardier",
    "Pilatus Aircraft Ltd": "Pilatus", "Socata - Groupe Aerospatiale": "Socata",
    "Douglas": "Mcdonnell Douglas", "Gates Learjet": "Learjet",
    "Ercoupe (Eng & Research Corp.)": "Ercoupe",
}
df["Make"] = df["Make"].replace(make_map)

# --- Impute missing Aircraft.Category ---
# About 56,000 rows have no category. For makes that ONLY ever appear as
# "Airplane" when the category is known (e.g. Cessna, Piper), we can safely
# assume the missing category is also "Airplane".
known = df.dropna(subset=["Aircraft.Category"])
make_categories = known.groupby("Make")["Aircraft.Category"].agg(set)
airplane_only_makes = make_categories[make_categories.apply(lambda s: s == {"Airplane"})].index

missing_cat = df["Aircraft.Category"].isna() & df["Make"].isin(airplane_only_makes)
df.loc[missing_cat, "Aircraft.Category"] = "Airplane"
print("Categories imputed as Airplane:", missing_cat.sum())

# --- Filter to what the client wants ---
df["Event.Date"] = pd.to_datetime(df["Event.Date"])
df = df[
    (df["Aircraft.Category"] == "Airplane")      # airplanes only
    & (df["Amateur.Built"] == "No")              # professional builds only
    & (df["Event.Date"].dt.year >= 1983)         # last 40 years (could still be active)
].copy()

print("Rows after filtering:", df.shape)

Aircraft.Category
NaN           56602
Airplane      27617
Helicopter     3440
Glider          508
Balloon         231
Name: count, dtype: int64 

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64 



Categories imputed as Airplane: 43490
Rows after filtering: (63817, 31)


**Filtering decisions:**
- `Make` names were standardized (title case, merged spelling variants) so each manufacturer is counted once.
- About 56,000 records had no `Aircraft.Category`. For manufacturers that only ever appear as "Airplane" when the category is known, the missing category was imputed as "Airplane". This keeps most of the usable data.
- Kept only airplanes, non-amateur (professional) builds, and events from 1983 onwards (max 40-year aircraft lifetime).

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:

# --- Construct metric for fatal/serious injuries ---
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries",
               "Total.Minor.Injuries", "Total.Uninjured"]

# Missing injury counts are assumed to be 0 (no one reported in that category)
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total people on board = everyone accounted for in the four categories
df["Total.Passengers"] = df[injury_cols].sum(axis=1)

# Remove records where no one is accounted for (can't compute a fraction)
df = df[df["Total.Passengers"] > 0].copy()

# Fraction of people on board who were fatally or seriously injured
df["Fatal.Serious.Fraction"] = (
    (df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]) / df["Total.Passengers"]
)

print("Rows:", df.shape)
df[["Total.Passengers", "Fatal.Serious.Fraction"]].describe()

Rows: (62851, 33)


,Total.Passengers,Fatal.Serious.Fraction
count,62851.000000,62851.000000
mean,4.588042,0.274445
std,21.812152,0.430685
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,0.750000
max,588.000000,1.000000


**Injury metric:**
- Missing injury counts were filled with 0 (assumed none reported).
- `Total.Passengers` estimates the number of people on board as the sum of fatal, serious, minor and uninjured counts. Records with 0 people accounted for were removed.
- `Fatal.Serious.Fraction` = (fatal + serious injuries) / total passengers. This measures the likelihood of a fatal or serious injury for a person on board, which is what the client wants to minimize.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:

# --- Aircraft.Damage ---
print(df["Aircraft.damage"].value_counts(dropna=False), "\n")

# "Unknown" and missing values can't tell us if the plane was destroyed, so we drop them
df = df[df["Aircraft.damage"].isin(["Destroyed", "Substantial", "Minor"])].copy()

# Derived column: 1 if the aircraft was destroyed, 0 otherwise
df["Destroyed"] = (df["Aircraft.damage"] == "Destroyed").astype(int)

print("Rows:", df.shape)
print("Share of accidents where the aircraft was destroyed:", round(df["Destroyed"].mean(), 3))

Aircraft.damage
Substantial    47098
Destroyed      12938
Minor           1431
NaN             1310
Unknown           74
Name: count, dtype: int64 



Rows: (61467, 34)
Share of accidents where the aircraft was destroyed: 0.21


**Aircraft damage:**
- Records with "Unknown" or missing damage were removed, since they can't indicate whether the aircraft was destroyed.
- `Destroyed` is a derived binary column: 1 if the aircraft was destroyed, 0 if damage was substantial or minor. Its mean gives the destruction rate for any group of aircraft.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Make cleaning tasks:**
1. Inconsistent capitalization (e.g. "CESSNA" vs "Cessna"): converted all names to title case and stripped extra spaces *(done in the filtering step, since the category imputation relied on it)*.
2. Same manufacturer under different names (e.g. "Cessna Aircraft Co", "Textron Aviation Inc" → "Cessna"; "Canadair" → "Bombardier"; "Raytheon Aircraft Company" → "Beech"): merged with a mapping dictionary *(also done in the filtering step)*.
3. Rows with a missing Make: dropped.
4. Makes with very few accidents give unreliable averages: kept only makes with at least 50 accidents, so comparisons between makes are statistically robust.

In [6]:
# --- Investigate and clean the Make column ---
# Drop missing makes
df = df.dropna(subset=["Make"])

# Look at the most common makes (after the standardization done earlier)
make_counts = df["Make"].value_counts()
print("Number of distinct makes:", make_counts.shape[0])
print(make_counts.head(20), "\n")

# Keep only makes with at least 50 accidents for robust comparisons
make_counts = df["Make"].value_counts()
df = df[df["Make"].isin(make_counts[make_counts >= 50].index)].copy()

print("Makes kept (>= 50 accidents):", df["Make"].nunique())
print("Rows:", df.shape)

Number of distinct makes: 1042
Make
Cessna            25424
Piper             13977
Beech              5121
Grumman            1499
Mooney             1290
Bellanca            973
Air Tractor         889
Aeronca             604
Champion            569
Maule               569
Boeing              481
De Havilland        460
Cirrus              431
Stinson             421
Aero Commander      405
Rockwell            403
Luscombe            386
North American      366
Taylorcraft         364
Ayres               273
Name: count, dtype: int64 

Makes kept (>= 50 accidents): 47
Rows: (57804, 34)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# --- Inspect the Model column ---
# Drop missing models
df = df.dropna(subset=["Model"])

# Standardize model labels (remove extra spaces, consistent upper case)
df["Model"] = df["Model"].str.strip().str.upper()

# Are model labels unique to each make?
makes_per_model = df.groupby("Model")["Make"].nunique()
shared_models = makes_per_model[makes_per_model > 1]
print("Model labels used by more than one make:", len(shared_models))
print(shared_models.head(10), "\n")

# Model names are NOT unique across makes, so create a unique plane-type identifier
df["Make.Model"] = df["Make"] + " " + df["Model"]

print("Distinct plane types (Make + Model):", df["Make.Model"].nunique())
df["Make.Model"].value_counts().head(10)

Model labels used by more than one make: 170
Model
100        3
100-180    2
112        3
112A       2
112TCA     2
145        2
180        2
19         2
20         2
200        3
Name: Make, dtype: int64 

Distinct plane types (Make + Model): 4094


Make.Model
Cessna 152         2210
Cessna 172         1615
Cessna 172N        1080
Piper PA-28-140     857
Cessna 172M         755
Cessna 150          742
Cessna 172P         661
Cessna 182          608
Cessna 180          593
Cessna 150M         550
Name: count, dtype: int64

**Model cleaning:**
- Missing models were dropped; model labels were stripped of spaces and converted to upper case for consistency.
- Some model labels are used by more than one manufacturer, so model alone does not identify a plane type.
- `Make.Model` was created by combining Make and Model, giving a unique identifier for each plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# --- Cleaning other columns ---
other_cols = ["Engine.Type", "Weather.Condition", "Number.of.Engines",
              "Purpose.of.flight", "Broad.phase.of.flight"]

# Inspect each column before cleaning
for col in other_cols:
    print(df[col].value_counts(dropna=False), "\n")

Engine.Type
Reciprocating      51270
NaN                 2875
Turbo Prop          2393
Turbo Fan            628
Unknown              408
Turbo Jet            200
Turbo Shaft           15
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

Weather.Condition
VMC    50960
IMC     4531
NaN     1671
UNK      492
Unk      137
Name: count, dtype: int64 

Number.of.Engines
1.0    48123
2.0     7364
NaN     2124
0.0       94
4.0       67
3.0       19
Name: count, dtype: int64 

Purpose.of.flight
Personal                     34637
Instructional                 8019
Aerial Application            3273
Unknown                       3091
Business                      2882
NaN                           2372
Positioning                    955
Ferry                          524
Other Work Use                 507
Aerial Observation             374
Executive/corporate            318
Public Aircraft                289
Skydiving                      167
Flight Test                

In [9]:
# Engine.Type: "Unknown", "UNK", "NONE" and "LR" are placeholders for missing values
df["Engine.Type"] = df["Engine.Type"].replace(
    {"Unknown": np.nan, "UNK": np.nan, "NONE": np.nan, "LR": np.nan})

# Weather.Condition: "Unk" and "UNK" are the same unknown placeholder -> missing
df["Weather.Condition"] = df["Weather.Condition"].str.upper().replace({"UNK": np.nan})

# Number.of.Engines: an airplane cannot have 0 engines -> treat as missing
df.loc[df["Number.of.Engines"] == 0, "Number.of.Engines"] = np.nan

# Purpose.of.flight: "Unknown" -> missing; group rare purposes (< 50 records) into "Other"
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({"Unknown": np.nan})
purpose_counts = df["Purpose.of.flight"].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df.loc[df["Purpose.of.flight"].isin(rare_purposes), "Purpose.of.flight"] = "Other"

# Broad.phase.of.flight: "Unknown" and "Other" carry no information -> missing
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace(
    {"Unknown": np.nan, "Other": np.nan})

# Check the results
for col in other_cols:
    print(df[col].value_counts(dropna=False), "\n")

Engine.Type
Reciprocating      51270
NaN                 3284
Turbo Prop          2393
Turbo Fan            628
Turbo Jet            200
Turbo Shaft           15
Geared Turbofan        1
Name: count, dtype: int64 

Weather.Condition
VMC    50960
IMC     4531
NaN     2300
Name: count, dtype: int64 

Number.of.Engines
1.0    48123
2.0     7364
NaN     2218
4.0       67
3.0       19
Name: count, dtype: int64 

Purpose.of.flight
Personal               34637
Instructional           8019
NaN                     5463
Aerial Application      3273
Business                2882
Positioning              955
Ferry                    524
Other Work Use           507
Aerial Observation       374
Executive/corporate      318
Public Aircraft          289
Skydiving                167
Other                    165
Flight Test              122
Banner Tow                96
Name: count, dtype: int64 

Broad.phase.of.flight
NaN            16098
Landing        11647
Takeoff         8620
Cruise          6960
Ma

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [10]:
# --- Column Removal ---
# Non-null count for each column
print(df.notna().sum().sort_values(), "\n")

# Keep only columns with more than 20,000 non-null values
cols_before = df.shape[1]
df = df.loc[:, df.notna().sum() > 20000]

print(f"Columns: {cols_before} -> {df.shape[1]}")
print("Final shape:", df.shape)

Schedule                   6066
Air.carrier                9407
FAR.Description           17200
Longitude                 21698
Latitude                  21704
Airport.Code              35254
Airport.Name              36900
Broad.phase.of.flight     41693
Publication.Date          47717
Purpose.of.flight         52328
Engine.Type               54507
Report.Status             54777
Weather.Condition         55491
Number.of.Engines         55573
Registration.Number       57275
Country                   57711
Location                  57768
Model                     57791
Aircraft.Category         57791
Aircraft.damage           57791
Injury.Severity           57791
Accident.Number           57791
Investigation.Type        57791
Event.Id                  57791
Event.Date                57791
Make                      57791
Total.Fatal.Injuries      57791
Amateur.Built             57791
Total.Uninjured           57791
Total.Minor.Injuries      57791
Total.Serious.Injuries    57791
Total.Pa

**Column removal:** columns with 20,000 or fewer non-null values (e.g. `Schedule`, `Air.carrier`) were dropped, since they are too sparse to support reliable analysis.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [11]:
# --- Save the cleaned data for the analysis notebook ---
df.to_csv("data/AviationData_cleaned.csv", index=False)
print("Saved:", df.shape)

Saved: (57791, 32)
